## Preiliminary Analysis

### Importing dependencies

In [0]:
import pyspark.sql.functions as sf

### Dataframe creation

In [0]:
df = (
    spark.read.option("header", "true")
    .option("inferSchema", "true")
    .option("sep", ",")
    .csv(
        "/Volumes/aml_transactions/aml_transactions_schema/source_files/HI-Small_Trans.csv"
    )
)

### Dataframe analysis

In [0]:
display(df.limit(5))

Timestamp,From Bank,Account2,To Bank,Account4,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
2022/09/01 00:20,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2022/09/01 00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
2022/09/01 00:02,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
2022/09/01 00:06,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


### Infered schema evaluation

In [0]:
df.printSchema()

root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)



### Total row count

In [0]:
df.count()


5078345

### Null check

In [0]:
df.filter(sf.col("Timestamp").isNull()).count()
df.filter(sf.col("From Bank").isNull()).count()
df.filter(sf.col("Account2").isNull()).count()
df.filter(sf.col("To Bank").isNull()).count()
df.filter(sf.col("Account4").isNull()).count()
df.filter(sf.col("Amount Received").isNull()).count()
df.filter(sf.col("Receiving Currency").isNull()).count()
df.filter(sf.col("Amount Paid").isNull()).count()
df.filter(sf.col("Payment Currency").isNull()).count()
df.filter(sf.col("Payment Format").isNull()).count()
df.filter(sf.col("Is Laundering").isNull()).count()

0

### Label distribution

In [0]:
df.filter(sf.col("Is Laundering") == 1).count()
df.filter(sf.col("Is Laundering") == 0).count()

5073168

### Timestamp range

In [0]:
df.select(sf.max("Timestamp")).show()

+----------------+
|  max(Timestamp)|
+----------------+
|2022/09/18 16:18|
+----------------+



In [0]:
df.select(sf.min("Timestamp")).show()

+----------------+
|  min(Timestamp)|
+----------------+
|2022/09/01 00:00|
+----------------+



### Amount ranges

In [0]:
df.select(sf.max("Amount Received")).show()

+--------------------+
|max(Amount Received)|
+--------------------+
| 1.04630236329348E12|
+--------------------+



In [0]:
df.select(sf.min("Amount Received")).show()

+--------------------+
|min(Amount Received)|
+--------------------+
|              1.0E-6|
+--------------------+



In [0]:
df.select(sf.max("Amount Paid")).show()

+-------------------+
|   max(Amount Paid)|
+-------------------+
|1.04630236329348E12|
+-------------------+



In [0]:
df.select(sf.min("Amount Paid")).show()

+----------------+
|min(Amount Paid)|
+----------------+
|          1.0E-6|
+----------------+



### Distinct categories

In [0]:
df.select("Payment Format").distinct().show()

+--------------+
|Payment Format|
+--------------+
|        Cheque|
|           ACH|
|       Bitcoin|
|          Wire|
|   Credit Card|
|          Cash|
|  Reinvestment|
+--------------+



In [0]:
df.select("Payment Currency").distinct().show()

+-----------------+
| Payment Currency|
+-----------------+
|     Mexican Peso|
|            Rupee|
|      Swiss Franc|
|           Shekel|
|             Yuan|
|             Euro|
|          Bitcoin|
|              Yen|
|            Ruble|
|         UK Pound|
|  Canadian Dollar|
|      Brazil Real|
|      Saudi Riyal|
|        US Dollar|
|Australian Dollar|
+-----------------+



In [0]:
df.select("Receiving Currency").distinct().show()

+------------------+
|Receiving Currency|
+------------------+
|      Mexican Peso|
|             Rupee|
|       Swiss Franc|
|            Shekel|
|              Yuan|
|              Euro|
|           Bitcoin|
|               Yen|
|             Ruble|
|          UK Pound|
|   Canadian Dollar|
|       Brazil Real|
|       Saudi Riyal|
|         US Dollar|
| Australian Dollar|
+------------------+



### Large transactions

In [0]:
max_amt_paid = df.select(sf.max("Amount Paid")).collect()[0][0]
df.filter(sf.col("Amount Paid") == max_amt_paid).show()

+----------------+---------+---------+-------+---------+-------------------+------------------+-------------------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|    Amount Received|Receiving Currency|        Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+-------------------+------------------+-------------------+----------------+--------------+-------------+
|2022/09/06 17:49|    19925|803D95360| 210185|803DA4620|1.04630236329348E12|               Yen|1.04630236329348E12|             Yen|           ACH|            0|
+----------------+---------+---------+-------+---------+-------------------+------------------+-------------------+----------------+--------------+-------------+



In [0]:
max_amt_received = df.select(sf.max("Amount Received")).collect()[0][0]
df.filter(sf.col("Amount Received") == max_amt_received).show()

+----------------+---------+---------+-------+---------+-------------------+------------------+-------------------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|    Amount Received|Receiving Currency|        Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+-------------------+------------------+-------------------+----------------+--------------+-------------+
|2022/09/06 17:49|    19925|803D95360| 210185|803DA4620|1.04630236329348E12|               Yen|1.04630236329348E12|             Yen|           ACH|            0|
+----------------+---------+---------+-------+---------+-------------------+------------------+-------------------+----------------+--------------+-------------+



### Duplicate check

In [0]:
tot_row_count = df.count()
print(tot_row_count)

5078345


In [0]:
dist_row_count = df.dropDuplicates().count()
print(dist_row_count)

5078336


In [0]:
dup_row_count = df.count() - df.dropDuplicates().count()
print(dup_row_count)

9


In [0]:
dup_pct = (dup_row_count/tot_row_count) * 100
print(dup_pct)

0.00017722309138114878


In [0]:
has_duplicates = (
    df.dropDuplicates().count() 
    > df.dropDuplicates(["Timestamp", "From Bank", "Account2", "To Bank", "Account4", "Amount Received", "Receiving Currency", "Amount Paid", "Payment Currency", "Payment Format"]).count()
    )
print(has_duplicates)

False


In [0]:
df.groupBy(
    "Timestamp",
    "From Bank",
    "Account2",
    "To Bank",
    "Account4",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
).agg(
    sf.count("*").alias("net_count"),
    sf.concat_ws(",", sf.collect_list("Is Laundering")).alias("laundering values"),
).filter(sf.col("net_count") > 1).show()

+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+---------+-----------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|net_count|laundering values|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+---------+-----------------+
|2022/09/09 10:03|     6075|80C702911|   6075|80C702910|         2.0E-6|           Bitcoin|       0.02|       US Dollar|           ACH|        2|              0,0|
|2022/09/09 21:33|    14433|80935A891|     15|813F7AE61|         1.0E-6|           Bitcoin|     1.0E-6|         Bitcoin|       Bitcoin|        2|              0,0|
|2022/09/08 21:05|   113779|811144AB1| 113779|811144AB0|         2.0E-6|           Bitcoin|       0.02|       US Dollar|           ACH|        2|              0,0|
|2022/09/09 21:3